In [1]:
import requests
import pandas as pd

In [2]:
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY not found in .env")

SEARCH_URL = "https://www.googleapis.com/youtube/v3/search"
VIDEOS_URL = "https://www.googleapis.com/youtube/v3/videos"
CHANNELS_URL = "https://www.googleapis.com/youtube/v3/channels"

In [3]:
keywords = [
    "python",
    "machine learning",
    "data science",
    "artificial intelligence",
    "deep learning"
]

In [4]:
def search_videos(keyword, max_results=5):
    """
    Search YouTube videos using a keyword.

    Parameters:
        keyword (str): Search term.
        max_results (int): Number of videos to retrieve.

    Returns:
        list: List of YouTube video IDs.
    """
    params = {
        "part": "snippet",
        "q": keyword,
        "type": "video",
        "maxResults": max_results,
        "key": API_KEY
    }

    response = requests.get(SEARCH_URL, params=params)

    response.raise_for_status()

    data = response.json()

    video_ids = []

    for item in data["items"]:
        video_id = item["id"].get("videoId")

        if video_id:
            video_ids.append(video_id)

    return video_ids

In [5]:
def get_video_details(video_ids):
    """
    Retrieve detailed information for a list of YouTube videos.

    Parameters:
        video_ids (list): List of YouTube video IDs.

    Returns:
        list: List of dictionaries containing video details.
    """

    video_ids = ",".join(video_ids)

    params = {
        "part": "snippet,statistics,contentDetails",
        "id": video_ids,
        "key": API_KEY
    }

    response = requests.get(VIDEOS_URL, params=params)

    response.raise_for_status()

    data = response.json()
    video_details = []
    for item in data["items"]:
      video = {
        "video_id": item["id"],
          "title": item["snippet"].get("title"),
          "description": item["snippet"].get("description"),
          "published_at": item["snippet"].get("publishedAt"),
          "channel_id": item["snippet"].get("channelId"),
          "channel_title": item["snippet"].get("channelTitle"),
          "category_id": item["snippet"].get("categoryId"),
          "tags": item["snippet"].get("tags"),
          "duration": item["contentDetails"].get("duration"),
          "view_count": item["statistics"].get("viewCount"),
          "like_count": item["statistics"].get("likeCount"),
          "comment_count": item["statistics"].get("commentCount"),
          "thumbnail_url": item["snippet"].get("thumbnails", {}).get("high", {}).get("url")
      }
      video_details.append(video)

    return video_details

In [6]:
def get_channel_details(channel_ids):
    """
    Retrieve detailed information for a list of YouTube channels.

    Parameters:
        channel_ids (list): List of YouTube channel IDs.

    Returns:
        list: List of dictionaries containing channel details.
    """

    channel_ids = ",".join(channel_ids)

    params = {
        "part": "snippet,statistics",
        "id": channel_ids,
        "key": API_KEY
    }
    response = requests.get(CHANNELS_URL, params=params)

    response.raise_for_status()
    data = response.json()
    channel_details = []

    for item in data["items"]:

        channel = {
            "channel_id": item["id"],
            "channel_title": item["snippet"].get("title"),
            "channel_description": item["snippet"].get("description"),
            "country": item["snippet"].get("country"),
            "published_at": item["snippet"].get("publishedAt"),
            "subscriber_count": item["statistics"].get("subscriberCount"),
            "video_count": item["statistics"].get("videoCount"),
            "view_count": item["statistics"].get("viewCount")
        }

        channel_details.append(channel)

    return channel_details

In [7]:
def merge_video_channel_data(videos, channels):
    """
    Merge video metadata with channel metadata.

    Parameters:
        videos (list): List of video dictionaries.
        channels (list): List of channel dictionaries.

    Returns:
        list: Combined video-channel dataset.
    """
    merged_data = []
    channel_lookup = {}

    for channel in channels:
      channel_lookup[channel["channel_id"]] = channel

    for video in videos:
      channel = channel_lookup.get(video["channel_id"])

      merged_record = {
            **video,
            **channel
        }

      merged_data.append(merged_record)

    return merged_data

In [8]:
master_dataset = []

for keyword in keywords:

    print(f"Collecting: {keyword}")

    video_ids = search_videos(keyword, max_results=20)

    videos = get_video_details(video_ids)

    channel_ids = [video["channel_id"] for video in videos]

    channels = get_channel_details(channel_ids)

    merged_data = merge_video_channel_data(videos, channels)

    master_dataset.extend(merged_data)

print(f"Collected {len(master_dataset)} records")

Collecting: python
Collecting: machine learning
Collecting: data science
Collecting: artificial intelligence
Collecting: deep learning
Collected 97 records


In [9]:
print(len(master_dataset))

97


In [10]:
import pandas as pd
df = pd.DataFrame(master_dataset)

In [11]:
df.to_csv("../data/raw/youtube_dataset.csv", index=False)

In [12]:
print(df.shape)
df.head()

(97, 17)


,video_id,title,description,published_at,channel_id,channel_title,category_id,tags,duration,view_count,like_count,comment_count,thumbnail_url,channel_description,country,subscriber_count,video_count
0,sxQnkMfp_6M,హైదరాబాద్‌ KBR పార్క్‌లో భారీ కొండచిలువ | Huge...,హైదరాబాద్‌ KBR పార్క్‌లో భారీ కొండచిలువ | Huge...,2014-03-29T06:02:22Z,UCDKjhgRoPF1CQk7HluMz23A,Mahaa News,25,"[Huge Python at Hyderabad KBR Park, Hyderabad ...",PT1M40S,3787894875,273,41,https://i.ytimg.com/vi/sxQnkMfp_6M/hqdefault.jpg,\nWatch : \nMahaa News Live is a 24-hour Telug...,IN,2860000,387957
1,_uQrJ0TkZlc,Python Full Course for Beginners,"Learn Python for AI, machine learning, and web...",2014-10-07T00:40:53Z,UCWv7vMbMWH4-V0ZXdmDpPBA,Programming with Mosh,27,"[python tutorial, python, python for beginners...",PT6H14M7S,275352417,1261583,62840,https://i.ytimg.com/vi/_uQrJ0TkZlc/hqdefault.jpg,"Hi! I'm Mosh 👋, a software engineer with over ...",US,5110000,246
2,q2-pnQffZik,Learn Python for FREE in 2025,Learn Python for FREE in 2025 #coding #compsci...,2020-06-18T00:50:20.154391Z,UC7zZ2-Q_oxbUaoMVL0z51wg,Sajjaad Khader,27,None,PT22S,61209740,30269,233,https://i.ytimg.com/vi/q2-pnQffZik/hqdefault.jpg,Teaching you how to break into tech in 2026. \...,US,323000,1320
3,yZl5FJ3ChkI,Python Roadmap For Beginners (Step By Step),"If I was a beginner learning to code, I would ...",2023-07-19T04:42:59.663153Z,UCgKFOz_KrMbmypWrawtzDQg,Erik Cupsa,28,None,PT53S,19270503,28242,141,https://i.ytimg.com/vi/yZl5FJ3ChkI/hqdefault.jpg,Welcome to the channel!\n\nThis is where I doc...,NaN,108000,744
4,K5KVEU3aaeQ,Python Full Course for Beginners,Master Python from scratch 🚀 No fluff—just cle...,2014-10-07T00:40:53Z,UCWv7vMbMWH4-V0ZXdmDpPBA,Programming with Mosh,27,"[python full course, python full course for be...",PT2H2M21S,275352417,159378,4170,https://i.ytimg.com/vi/K5KVEU3aaeQ/hqdefault.jpg,"Hi! I'm Mosh 👋, a software engineer with over ...",US,5110000,246
